In [2]:
"""
Folder Size Treemap Visualizer
Analyzes a directory one level deep and creates a treemap of folder sizes
"""

import os
import matplotlib.pyplot as plt
import squarify

# ---------- Size formatter ----------
def format_size(size_bytes):
    """Convert bytes to human-readable format"""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024
    return f"{size_bytes:.2f} PB"

# ---------- Color palette ----------
TREEMAP_COLORS = [
    "#4C72B0", "#55A868", "#C44E52", "#8172B3", "#CCB974",
    "#64B5CD", "#8C8C8C", "#E17C05", "#76B7B2", "#F28E2B",
    "#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"
]

# ---------- Calculate folder size ----------
def get_folder_size(folder_path):
    """Calculate total size of all files in a folder (recursive)"""
    total_size = 0
    try:
        for dirpath, dirnames, filenames in os.walk(folder_path):
            for filename in filenames:
                filepath = os.path.join(dirpath, filename)
                try:
                    total_size += os.path.getsize(filepath)
                except (OSError, FileNotFoundError):
                    # Skip files that can't be accessed
                    pass
    except (OSError, PermissionError):
        # Skip folders that can't be accessed
        pass
    return total_size

# ---------- Analyze directory one level deep ----------
def analyze_directory(parent_path):
    """
    Analyze directory one level deep and return folder sizes
    Returns: dict of {folder_name: size_in_bytes}
    """
    if not os.path.exists(parent_path):
        print(f"❌ Error: Path does not exist: {parent_path}")
        return {}
    
    if not os.path.isdir(parent_path):
        print(f"❌ Error: Path is not a directory: {parent_path}")
        return {}
    
    folder_sizes = {}
    
    print(f"📂 Analyzing directory: {parent_path}")
    print("⏳ Calculating folder sizes...")
    
    try:
        items = os.listdir(parent_path)
        
        # Filter for directories only
        folders = [item for item in items if os.path.isdir(os.path.join(parent_path, item))]
        
        if not folders:
            print("❌ No subdirectories found in this directory.")
            return {}
        
        # Calculate size for each folder
        for folder in folders:
            folder_path = os.path.join(parent_path, folder)
            size = get_folder_size(folder_path)
            folder_sizes[folder] = size
            print(f"  ✓ {folder}: {format_size(size)}")
        
        return folder_sizes
        
    except PermissionError:
        print(f"❌ Permission denied: Cannot access {parent_path}")
        return {}

# ---------- Create and save treemap ----------
def create_folder_treemap(folder_sizes, parent_path, output_path=None):
    """
    Create treemap visualization of folder sizes
    """
    if not folder_sizes:
        print("❌ No folder data to plot.")
        return
    
    # Sort by size (largest first) for better visualization
    sorted_items = sorted(folder_sizes.items(), key=lambda x: x[1], reverse=True)
    
    labels = []
    sizes = []
    
    for folder_name, size in sorted_items:
        if size > 0:  # Only include non-empty folders
            labels.append(folder_name)
            sizes.append(size)
    
    if not sizes:
        print("❌ No non-empty folders found.")
        return
    
    # Calculate total size and percentages
    total_size = sum(sizes)
    
    print(f"\n📊 Total size: {format_size(total_size)}")
    print(f"📁 Number of folders: {len(sizes)}")
    
    # Label threshold - show full details only for folders > 2% of total
    LABEL_THRESHOLD_PERCENT = 2.0
    
    # Create display labels
    display_labels = []
    for label, size in zip(labels, sizes):
        percentage = (size / total_size) * 100
        if percentage >= LABEL_THRESHOLD_PERCENT:
            display_labels.append(f"{label}\n{format_size(size)}\n({percentage:.1f}%)")
        else:
            display_labels.append(f"{label}")
    
    # Assign colors
    colors = (TREEMAP_COLORS * ((len(sizes) // len(TREEMAP_COLORS)) + 1))[:len(sizes)]
    
    # Create figure
    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(111)
    
    squarify.plot(
        sizes=sizes,
        label=display_labels,
        color=colors,
        alpha=0.88,
        text_kwargs={'fontsize': 9, 'fontweight': 'bold', 'wrap': True},
        ax=ax
    )
    
    # Set title
    parent_name = os.path.basename(parent_path.rstrip(os.sep)) or parent_path
    ax.set_title(
        f"Folder Size Distribution\n{parent_name}",
        fontsize=16,
        fontweight="bold",
        pad=20
    )
    ax.axis("off")
    
    # Add legend for small folders
    small_folders = [
        (label, size) for label, size in zip(labels, sizes)
        if (size / total_size) * 100 < LABEL_THRESHOLD_PERCENT
    ]
    
    if small_folders:
        legend_text = "Small folders:\n" + "\n".join([
            f"{folder}: {format_size(size)} ({(size/total_size)*100:.1f}%)"
            for folder, size in small_folders
        ])
        
        ax.text(
            1.02, 0.5, legend_text,
            transform=ax.transAxes,
            fontsize=8,
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3)
        )
    
    # Save the plot
    if output_path is None:
        output_path = os.path.join(parent_path, "folder_size_treemap.png")
    
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    
    print(f"\n✅ Treemap saved: {output_path}")

# ---------- Print summary table ----------
def print_summary_table(folder_sizes):
    """Print a formatted table of folder sizes"""
    if not folder_sizes:
        return
    
    sorted_items = sorted(folder_sizes.items(), key=lambda x: x[1], reverse=True)
    total_size = sum(folder_sizes.values())
    
    print("\n" + "="*70)
    print(f"{'FOLDER NAME':<40} {'SIZE':<15} {'PERCENTAGE':>10}")
    print("="*70)
    
    for folder_name, size in sorted_items:
        percentage = (size / total_size) * 100
        print(f"{folder_name:<40} {format_size(size):<15} {percentage:>9.2f}%")
    
    print("="*70)
    print(f"{'TOTAL':<40} {format_size(total_size):<15} {'100.00%':>10}")
    print("="*70)

# ========================================
# MAIN FUNCTION
# ========================================
def visualize_folder_sizes(directory_path, output_path=None):
    """
    Main function to analyze and visualize folder sizes
    
    Args:
        directory_path: Path to the parent directory to analyze
        output_path: Optional path to save the treemap image
    """
    # Analyze directory
    folder_sizes = analyze_directory(directory_path)
    
    if not folder_sizes:
        return
    
    # Print summary table
    print_summary_table(folder_sizes)
    
    # Create treemap
    create_folder_treemap(folder_sizes, directory_path, output_path)

# ========================================
# USAGE EXAMPLE
# ========================================
if __name__ == "__main__":
    # Example usage - replace with your directory path
    
    # Option 1: Provide directory path directly
    directory_path = "/path/to/your/directory"
    visualize_folder_sizes(directory_path)
    
    # Option 2: Provide both directory and output path
    # directory_path = "/path/to/your/directory"
    # output_path = "/path/to/save/treemap.png"
    # visualize_folder_sizes(directory_path, output_path)
    
    # Option 3: Get input from user
    # directory_path = input("Enter directory path: ")
    # visualize_folder_sizes(directory_path)

❌ Error: Path does not exist: /path/to/your/directory


In [3]:
# Simple usage
visualize_folder_sizes("/Volumes/PPS64/well data/Australia")


📂 Analyzing directory: /Volumes/PPS64/well data/Australia
⏳ Calculating folder sizes...
  ✓ D00020236: 87.32 MB
  ✓ D00021233: 238.35 MB
  ✓ D00019986: 243.10 MB
  ✓ D00020952: 253.02 MB
  ✓ D00020079: 265.78 MB
  ✓ D00020052: 268.12 MB
  ✓ D00019997: 2.43 GB
  ✓ D00020029: 4.74 GB
  ✓ D00020186: 2.35 GB
  ✓ D00020558: 4.74 GB
  ✓ D00019225: 48.74 MB
  ✓ D00020495: 320.01 MB
  ✓ D00020715: 982.54 MB
  ✓ D00020042: 327.99 MB
  ✓ D00020074: 1.37 GB
  ✓ D00020260: 367.04 MB
  ✓ D00019914: 419.03 MB
  ✓ D00020055: 440.57 MB
  ✓ D00021115: 474.53 MB
  ✓ D00021001: 495.80 MB
  ✓ D00020198: 505.17 MB
  ✓ D00020000: 1.47 GB
  ✓ D00019269: 180.22 MB
  ✓ D00019663: 46.86 MB
  ✓ D00020189: 682.91 MB
  ✓ D00020367: 1.90 GB
  ✓ D00019666: 164.82 MB
  ✓ D00019726: 2.44 GB
  ✓ D00019815: 1.32 GB
  ✓ D00019867: 804.30 MB
  ✓ D00019841: 903.37 MB
  ✓ D00020193: 2.22 GB
  ✓ D00019505: 439.91 MB
  ✓ D00020185: 170.72 MB
  ✓ D00020039: 380.57 MB
  ✓ D00020128: 89.36 MB
  ✓ D00020227: 229.08 MB
  ✓ D000203